In [5]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# Load in Account A data
df = pd.read_csv("account_a.csv")

# Convert Datetime variable so it is parsed correctly
df["Datetime"] = pd.to_datetime(df["Datetime"], format="mixed", errors="coerce")

# Extract date and hour
df["date_only"] = df["Datetime"].dt.date
df["hour"] = df["Datetime"].dt.hour

# First 2 weeks (14 days) only
first_14_days = sorted(df["date_only"].unique())[:14]
df_14 = df[df["date_only"].isin(first_14_days)].copy()

# Map days and hours indexes
day_map = {day: i + 1 for i, day in enumerate(first_14_days)}
df_14["d"] = df_14["date_only"].map(day_map)
df_14["t"] = df_14["hour"]

# Calculate Price and Quantity
hourly = (
    df_14.groupby(["d", "t"], as_index=False)
    .agg({
        "Quantity": "sum",
        "Settlement.Point.Price.KWH": "mean"
    })
)

Price = {(row["d"], row["t"]): row["Settlement.Point.Price.KWH"] for _, row in hourly.iterrows()}
Quantity = {(row["d"], row["t"]): row["Quantity"] for _, row in hourly.iterrows()}

# Create set indexes
D = sorted(hourly["d"].unique())
T = sorted(hourly["t"].unique())
DT = sorted(list(Quantity.keys()))  

# Max batteries for 2 week period
B_max = 30

# Battery cost
C_battery = 11959

# Charging/Discharging rate per battery per hour
r_ch = 5
r_dis = 11.5

m = gp.Model("battery_model_account_a")

# Decision Variables
x_A = m.addVar(vtype=GRB.INTEGER, lb=0, ub=B_max)
e = m.addVars(range(1, B_max + 1), vtype=GRB.BINARY)
y = m.addVars(range(1, B_max + 1), DT, vtype=GRB.BINARY)
charge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
discharge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
cs = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)

# Constraints
# sum_b e_A,b = x_A
m.addConstr(
    gp.quicksum(e[b] for b in range(1, B_max + 1)) == x_A,
    name="num_active_batteries"
)

# y_b,A,t,d <= e_A,b
for b in range(1, B_max + 1):
    for d, t in DT:
        m.addConstr(y[b, d, t] <= e[b])

# g_{A,t,d} = Quantity_{A,t,d} 
for d, t in DT: 
    m.addConstr(discharge[d, t] == Quantity[d, t])
    
# charging state capacity constraints
for d, t in DT:
    m.addConstr(cs[d, t] <= 13.5 * x_A)
    m.addConstr(cs[d, t] >= 10.8 * x_A)


# charging/discharging rate limits
for d, t in DT:
    m.addConstr(charge[d, t] <= r_ch * x_A)
    m.addConstr(discharge[d, t] <= r_dis * x_A)

# Charging state across time
DT_sorted = sorted(DT)   

# Charging state constraints
for i, (d, t) in enumerate(DT_sorted):
    if i == 0:
        m.addConstr(
            cs[d, t] == 13.5 * x_A + charge[d, t] - discharge[d, t]
        )
    else:
        prev_d, prev_t = DT_sorted[i - 1]
        m.addConstr(
            cs[d, t] == cs[prev_d, prev_t] + charge[d, t] - discharge[d, t]
        )
    
# Objective Function
energy_cost = gp.quicksum(
    charge[d, t] * Price[d, t]
    for d, t in DT
)

battery_cost = x_A * C_battery

m.setObjective(energy_cost + battery_cost, GRB.MINIMIZE)

# Optimize
m.optimize()

# Print results
if m.status == GRB.OPTIMAL:
    print("\nOptimal Solution")
    print(f"x_A = {x_A.X}")
    print(f"Energy cost = {energy_cost.getValue():,.4f}")
    print(f"Battery cost = {battery_cost.getValue():,.4f}")
    print(f"Total objective = {m.objVal:,.4f}")

    # Display charge/discharge values for some iterations just to check
    print("\nSample hourly values:")
    for d, t in DT_sorted[:10]:
        print(
            f"(d={d}, t={t}) | charge={charge[d,t].X:.4f}, "
            f"discharge={discharge[d,t].X:.4f}, charge_state={cs[d,t].X:.4f}"
        )
else:
    print("No optimal solution found.")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 155U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 8317 rows, 7654 columns and 16894 nonzeros
Model fingerprint: 0xc372cf6d
Variable types: 693 continuous, 6961 integer (6960 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [6e-03, 1e+04]
  Bounds range     [1e+00, 3e+01]
  RHS range        [1e+00, 5e+00]
Found heuristic solution: objective 11997.055485
Presolve removed 7550 rows and 7345 columns
Presolve time: 0.01s
Presolved: 767 rows, 309 columns, 2070 nonzeros
Variable types: 308 continuous, 1 integer (0 binary)

Root relaxation: objective 1.199461e+04, 233 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/No